# 01. 온통청년 정책 데이터 전처리: 검증 반영 개선본

이 노트북은 `youth_policies_categorized.csv`를 분석용 정책 데이터셋으로 정리합니다.

## 핵심 전처리 기준

| 구분 | 처리 기준 |
|---|---|
| 분석 단위 | `지역 × 정책ID`를 기본 분석 단위로 사용 |
| 중복 제거 | 동일 지역 내 동일 정책ID만 제거. 단, 정책ID가 없으면 동일 지역 내 동일 정책명 기준 제거 |
| 지역 간 중복 | 동일 정책ID가 여러 지역에 등장하는 경우 전국/복수지역 대상 정책일 수 있으므로 제거하지 않음 |
| 분류 기준 | `정책대분류/정책중분류`를 우선 사용하고, 부족할 때 `대표분류/자동분류` 보조 사용 |
| 텍스트 결합 | 정책명, 키워드, 설명, 지원내용, 지원대상, 신청방법, 기관 정보를 결합하되 중복 문구와 코드성 잡음 제거 |
| 날짜 처리 | `YYYYMMDD`, `YYYY-MM-DD`, `YYYY. M. D.`, `YYYY년 M월 D일` 형식 일부 지원 |
| 산출물 | 전처리 완료 CSV, 지역 요약, 분류 요약, 지역×분류 피벗표, 전처리 품질 점검표 |

> 개선 포인트: 기존 노트북은 전체적으로 실행 가능한 구조였지만, `정책중분류=창업`인 정책이 `대표분류=일자리`로 남는 문제가 커서 창업지원 정책이 과소 집계될 수 있었습니다. 또한 텍스트 결합에서 `지원대상`, `나이조건`, `소득조건`이 중복 반영되어 TF-IDF/토픽모델링에 잡음이 생길 수 있어 개선했습니다.

In [1]:
from pathlib import Path
import re
import html
import calendar
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)

# Colab/VSCode/로컬에서 모두 찾을 수 있도록 후보 경로를 둡니다.
CANDIDATE_DIRS = [
    Path("."),
    Path("/content"),
    Path("/content/TM/tm"),
    Path("/content/drive/MyDrive/TM/tm"),
    Path("/mnt/data"),
]

POLICY_FILENAME = "youth_policies_categorized.csv"
SUMMARY_FILENAME = "youth_policies_summary_by_region_category.csv"


def find_file(filename: str) -> Path:
    for base in CANDIDATE_DIRS:
        path = base / filename
        if path.exists():
            return path

    # Colab에서 폴더 위치가 다를 때 /content 아래를 한 번 더 탐색합니다.
    for root in [Path("/content"), Path("/mnt/data")]:
        if root.exists():
            matches = list(root.rglob(filename))
            if matches:
                return matches[0]

    raise FileNotFoundError(
        f"{filename} 파일을 찾지 못했습니다. 노트북과 같은 폴더 또는 /content/drive/MyDrive/TM/tm에 넣어 주세요."
    )

POLICY_FILE = find_file(POLICY_FILENAME)
DATA_DIR = POLICY_FILE.parent
SUMMARY_FILE = DATA_DIR / SUMMARY_FILENAME
OUTPUT_DIR = DATA_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("정책 파일:", POLICY_FILE)
print("요약 파일 존재:", SUMMARY_FILE.exists())
print("저장 폴더:", OUTPUT_DIR)

정책 파일: youth_policies_categorized.csv
요약 파일 존재: True
저장 폴더: outputs


In [2]:
def read_csv_auto(path: Path) -> pd.DataFrame:
    """CSV 인코딩이 달라도 최대한 안정적으로 읽는 함수."""
    encodings = ["utf-8-sig", "utf-8", "cp949", "euc-kr"]
    last_error = None

    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception as e:
            last_error = e

    raise last_error


df_raw = read_csv_auto(POLICY_FILE)
summary_raw = read_csv_auto(SUMMARY_FILE) if SUMMARY_FILE.exists() else None

print("원본 정책 데이터 크기:", df_raw.shape)
display(df_raw.head())

원본 정책 데이터 크기: (5470, 27)


,지역,조회_zipCd,정책ID,정책명,정책키워드,정책설명,정책지원내용,정책대분류,정책중분류,자동분류,대표분류,신청기간,사업기간,지원대상,나이조건,소득조건,신청방법,제출서류,주관기관,운영기관,신청URL,참고URL1,참고URL2,최초등록일시,최종수정일시,수집페이지,수집출처
0,서울,11000,20260605005400113228,청년미래적금,보조금,"청년들의 기초자산 형성을 지원하기 위한 정책형 금융상품으로, 3년 만기 시까지 매월 최대 50만원 한도 내에서 자유롭게 납입 가능(2026년 6월 22일 출시 예정)","은행이자+비과세 혜택+정부기여금(납입금액에 비례해 일반형 6%, 우대형 12%의 정부기여금 지원)",금융･복지･문화,취약계층 및 금융지원,복지,복지,20260622 ~ 20261231,20260622 ~ 20261231,"19세~34세 / 연령제한:N 0043003 0 0 총급여 7,500만원 이하 또는 연매출 3억원 이하 소상공인 중 가구 중위소득 200% 이하인 청년을 대상 0011009 0013010 0049010 005...",19세~34세 / 연령제한:N,"0043003 0 0 총급여 7,500만원 이하 또는 연매출 3억원 이하 소상공인 중 가구 중위소득 200% 이하인 청년을 대상",ㅇ 취급은행 모바일앱을 통해 매월 비대면 신청 가능 ㅇ 2026년 6월 출시 예정,NaN,금융위원회,한국고용정보원,NaN,https://www.kinfa.or.kr/financialProduct/youthFutureSavings.do,https://blog.naver.com/blogfsc/224302863400,2026-06-05 18:06:49,2026-06-10 14:05:31,1,온통청년_OPEN_API_getPlcy
1,서울,11000,20260528005400113227,(농식품부) 농식품 바우처,바우처,『농업·농촌 및 식품산업 기본법』 제 23조의 2(취약계층 등에 대한 식품지원)에 근거하여 취약계층의 식품 접근성을 강화하고 균형 있는 식품 섭취를 지원하는 식품지원 제도,"- 지원방식: 전자바우처(카드방식) - 지원품목: 국산 채소류, 과일류, 육류, 신선알류, 흰 우유, 잡곡류, 두부류, 임산물 (이외 품목 구매 불가)",금융･복지･문화,건강,복지,복지,20251222 ~ 20261211,20260102 ~ 20261231,18세~34세 / 연령제한:N 0043003 0 0 생계급여(기준 중위소득 32%이하) 수급가구 중 임산부·영유아·아동·청년 포함가구 생계급여 수급가구 가구원 중 「국민기초생활 보장법」상 보장시설 수급자는 가...,18세~34세 / 연령제한:N,0043003 0 0 생계급여(기준 중위소득 32%이하) 수급가구 중 임산부·영유아·아동·청년 포함가구,1. 방문신청 : 주소지 관할 읍ㆍ면ㆍ동 행정복지센터 방문 2. 전화신청 : 고객지원센터 (1551-0857)를 통해 신청 3. 온라인신청 : 농식품 바우처 홈페이지에서 신청 4. 자동신청 : 2025년 농식...,NaN,농림축산식품부,한국고용정보원,https://www.foodvoucher.go.kr/security/joinAgree,https://www.foodvoucher.go.kr/view/fm/vucintro/agriFood,NaN,2026-05-28 10:00:50,2026-05-28 10:01:12,1,온통청년_OPEN_API_getPlcy
2,서울,11000,20260528005400113226,(문체부) 청년예술인 예술활동 적립계좌,보조금,청년예술인에게 중장기 자산형성의 기회를 마련하여 안정적인 예술활동을 지원하는 사업,매월 일정 금액을 24개월간 적금 저축 시 가입자가 저축한 금액만큼 정부지원금을 지원 (1인 최대 240만원) - 상품종류: 10만원 정액 적금 - 가입기간: 2년 (24개월) - 납입한도: 월 10만원 (2...,금융･복지･문화,예술인지원,복지,복지,NaN,20260101 ~ 20261231,"18세~39세 / 연령제한:N 0043002 0 3692 「예술인복지법」상 예술활동증명을 완료한 예술인(신청일 기준 예술활동증명 유효자) - 일반 예술활동증명 완료자(공개 발표된 예술활동, 예술활동 수입, 경...",18세~39세 / 연령제한:N,0043002 0 3692,1. 예술활동증명확인 예술인경력정보시스템(https://www.kawfartist.kr)을 접속하시어 경력지원 > 예술활동증명 > 신청내역 진행 상태에서 신청일 현재 예술활동증명 유효 여부를 확인합니다. 2....,1. 주민등록초본 - 2026년 발급분 - 발급 시 발급대상자 본인 및 전체 발급 2. 소득금액증명원 - 2026년 발급분 - 귀속년도: 직전년도(2024년도) 소득금액증명원 발급 - 작성기준: 종합소득세 신...,문화체육관광부,한국고용정보원,https://www.artloan.kr/notice/savingsAccountProcess.do,https://www.artloan.kr/notice/savingsAccount.do,NaN,2026-05-28 09:34:16,2026-05-28 09:34:49,1,온통청년_OPEN_API_getPlcy
3,서울,11000,20260527005400113224,청년 국가기술자격 응시료 지원 사업,보조금,"구직활동을 하거나 경력을 개발하는 청년들의 경제적 부담을 완화하고, 국가기술자격 취득을 통한 취업 경쟁력 강화를 도모","34세 이하 청년*이(소득 및 취업 여부 무관) 한국산업인력공단이 시행하는 국가기술자격 시험(’26년 기준 491종목)에 응시하는 경우 응시료의 50%를 선 지원 (1인당 年 3회 지원, 단 예산 242억 원...",일자리,취업,일자리,일자리,NaN,상시,0세~34세 / 연령제한:N 0043001 0 0 0011009 0013010 0049010 0055003,0세~34세 / 연령제한:N,0043001 0 0,"한국산업인력공단 Q-Net(https://www.q-net.or.kr) 원서접수 결제 단계에서, 34세 이하 청년인 경우 별도의 신청 절차 없이 50% 할인 자동 적용",NaN,고용노동부,한국산업인력공단,https://www.q-net.or.kr/man001.do?gSite=Q&gIntro=Y,https://www.q-net.or.kr/man004.do?id=man00402&gSite=Q&gId=,NaN,2026-05-27 11:16:43,2026-05-27 11:17:06,1,온통청년_OPEN_API_getPlcy
4,서울,11000,20260527005400113223,전세보증금반환보증 보증료 지원,주거지원,"전세사기, 역전세 등 임차인이 전세보증금을 돌려받지 못하는 전세 피해를 예방하고, 상대적으로 주거 취약계층인 청년 및 저소득층의 주거 안정을 도모","○ 지원대상 - 신청일 기준 유효한 전세보증금반환보증(HUG, HF, SGI)에 가입한 임차보증금 3억원 이하, 연소득 (청년) 5천만원, (청년외) 6천만원, (신혼부부) 7.5천만원 이하 무주택 임차인 ○...",주거,전월세 및 주거급여 지원,"주거지원, 복지",주거지원,NaN,상시,0세~0세 / 연령제한:Y 0043001 0 0 0011009 0013010 0049010 0055003,0세~0세 / 연령제한:Y,0043001 0 0,○ 신청방법 : 시·군·구청 또는 주민센터에 방문 접수 혹은 온라인 접수 ※ 방문접수의 경우 구군별 상이하니 전화 후 방문 필요 * 대구광역시는 온라인 접수 또는 시청 방문 신청 ○ 지급시기 및 지급방법 - ...,"○ 보증기관에서 보증가입 시 전세보증금반환보증 보증료 지원 사업을 위한 제3자 정보제공에 동의한 경우 - 보증료 지원 신청서, 서약서 ○ 신청인 제출 서류 - 보증료 지원 신청 및 서약서, 지급받을 계좌의 통...",국토교통부,국토교통부,https://www.gov.kr/portal/rcvfvrSvc/dtlEx/161300000103,https://www.gov.kr/portal/rcvfvrSvc/dtlEx/161300000103,NaN,2026-05-27 11:06:12,2026-05-27 11:06:36,1,온통청년_OPEN_API_getPlcy


In [3]:
# 필수 컬럼 보정
required_cols = [
    "지역", "조회_zipCd", "정책ID", "정책명", "정책키워드", "정책설명", "정책지원내용",
    "정책대분류", "정책중분류", "자동분류", "대표분류", "신청기간", "사업기간",
    "지원대상", "나이조건", "소득조건", "신청방법", "제출서류",
    "주관기관", "운영기관", "신청URL", "참고URL1", "참고URL2",
    "최초등록일시", "최종수정일시", "수집페이지", "수집출처"
]

df = df_raw.copy()
for col in required_cols:
    if col not in df.columns:
        df[col] = np.nan


def clean_basic_text(x) -> str:
    """결측, HTML 엔티티, 과도한 공백을 정리하는 기본 텍스트 정리 함수."""
    if pd.isna(x):
        return ""
    x = str(x)
    if x.strip().lower() in {"nan", "none", "null", "nat"}:
        return ""
    x = html.unescape(x)
    x = re.sub(r"<[^>]+>", " ", x)
    x = x.replace("\u200b", " ").replace("\xa0", " ")
    x = re.sub(r"\s+", " ", x)
    return x.strip()


text_like_cols = [c for c in required_cols if c != "수집페이지"]
for col in text_like_cols:
    df[col] = df[col].map(clean_basic_text)

print("컬럼 보정 후 크기:", df.shape)
display(df.head(3))

컬럼 보정 후 크기: (5470, 27)


,지역,조회_zipCd,정책ID,정책명,정책키워드,정책설명,정책지원내용,정책대분류,정책중분류,자동분류,대표분류,신청기간,사업기간,지원대상,나이조건,소득조건,신청방법,제출서류,주관기관,운영기관,신청URL,참고URL1,참고URL2,최초등록일시,최종수정일시,수집페이지,수집출처
0,서울,11000,20260605005400113228,청년미래적금,보조금,"청년들의 기초자산 형성을 지원하기 위한 정책형 금융상품으로, 3년 만기 시까지 매월 최대 50만원 한도 내에서 자유롭게 납입 가능(2026년 6월 22일 출시 예정)","은행이자+비과세 혜택+정부기여금(납입금액에 비례해 일반형 6%, 우대형 12%의 정부기여금 지원)",금융･복지･문화,취약계층 및 금융지원,복지,복지,20260622 ~ 20261231,20260622 ~ 20261231,"19세~34세 / 연령제한:N 0043003 0 0 총급여 7,500만원 이하 또는 연매출 3억원 이하 소상공인 중 가구 중위소득 200% 이하인 청년을 대상 0011009 0013010 0049010 005...",19세~34세 / 연령제한:N,"0043003 0 0 총급여 7,500만원 이하 또는 연매출 3억원 이하 소상공인 중 가구 중위소득 200% 이하인 청년을 대상",ㅇ 취급은행 모바일앱을 통해 매월 비대면 신청 가능 ㅇ 2026년 6월 출시 예정,,금융위원회,한국고용정보원,,https://www.kinfa.or.kr/financialProduct/youthFutureSavings.do,https://blog.naver.com/blogfsc/224302863400,2026-06-05 18:06:49,2026-06-10 14:05:31,1,온통청년_OPEN_API_getPlcy
1,서울,11000,20260528005400113227,(농식품부) 농식품 바우처,바우처,『농업·농촌 및 식품산업 기본법』 제 23조의 2(취약계층 등에 대한 식품지원)에 근거하여 취약계층의 식품 접근성을 강화하고 균형 있는 식품 섭취를 지원하는 식품지원 제도,"- 지원방식: 전자바우처(카드방식) - 지원품목: 국산 채소류, 과일류, 육류, 신선알류, 흰 우유, 잡곡류, 두부류, 임산물 (이외 품목 구매 불가)",금융･복지･문화,건강,복지,복지,20251222 ~ 20261211,20260102 ~ 20261231,18세~34세 / 연령제한:N 0043003 0 0 생계급여(기준 중위소득 32%이하) 수급가구 중 임산부·영유아·아동·청년 포함가구 생계급여 수급가구 가구원 중 「국민기초생활 보장법」상 보장시설 수급자는 가...,18세~34세 / 연령제한:N,0043003 0 0 생계급여(기준 중위소득 32%이하) 수급가구 중 임산부·영유아·아동·청년 포함가구,1. 방문신청 : 주소지 관할 읍ㆍ면ㆍ동 행정복지센터 방문 2. 전화신청 : 고객지원센터 (1551-0857)를 통해 신청 3. 온라인신청 : 농식품 바우처 홈페이지에서 신청 4. 자동신청 : 2025년 농식...,,농림축산식품부,한국고용정보원,https://www.foodvoucher.go.kr/security/joinAgree,https://www.foodvoucher.go.kr/view/fm/vucintro/agriFood,,2026-05-28 10:00:50,2026-05-28 10:01:12,1,온통청년_OPEN_API_getPlcy
2,서울,11000,20260528005400113226,(문체부) 청년예술인 예술활동 적립계좌,보조금,청년예술인에게 중장기 자산형성의 기회를 마련하여 안정적인 예술활동을 지원하는 사업,매월 일정 금액을 24개월간 적금 저축 시 가입자가 저축한 금액만큼 정부지원금을 지원 (1인 최대 240만원) - 상품종류: 10만원 정액 적금 - 가입기간: 2년 (24개월) - 납입한도: 월 10만원 (2...,금융･복지･문화,예술인지원,복지,복지,,20260101 ~ 20261231,"18세~39세 / 연령제한:N 0043002 0 3692 「예술인복지법」상 예술활동증명을 완료한 예술인(신청일 기준 예술활동증명 유효자) - 일반 예술활동증명 완료자(공개 발표된 예술활동, 예술활동 수입, 경...",18세~39세 / 연령제한:N,0043002 0 3692,1. 예술활동증명확인 예술인경력정보시스템(https://www.kawfartist.kr)을 접속하시어 경력지원 > 예술활동증명 > 신청내역 진행 상태에서 신청일 현재 예술활동증명 유효 여부를 확인합니다. 2....,1. 주민등록초본 - 2026년 발급분 - 발급 시 발급대상자 본인 및 전체 발급 2. 소득금액증명원 - 2026년 발급분 - 귀속년도: 직전년도(2024년도) 소득금액증명원 발급 - 작성기준: 종합소득세 신...,문화체육관광부,한국고용정보원,https://www.artloan.kr/notice/savingsAccountProcess.do,https://www.artloan.kr/notice/savingsAccount.do,,2026-05-28 09:34:16,2026-05-28 09:34:49,1,온통청년_OPEN_API_getPlcy


In [4]:
# 중복 구조 진단: 제거 전 반드시 유형별 중복을 확인합니다.
df["정책ID_정리"] = df["정책ID"].map(clean_basic_text)
df["정책명_정리"] = df["정책명"].map(clean_basic_text)
df["지역_정리"] = df["지역"].map(clean_basic_text)

duplicate_report = pd.DataFrame({
    "점검기준": [
        "행 전체 완전 동일",
        "동일 지역 + 동일 정책ID",
        "동일 정책ID만 기준",
        "동일 지역 + 동일 정책명",
        "동일 정책명만 기준",
    ],
    "중복행수": [
        int(df.duplicated().sum()),
        int(df.duplicated(subset=["지역_정리", "정책ID_정리"]).sum()),
        int(df.duplicated(subset=["정책ID_정리"]).sum()),
        int(df.duplicated(subset=["지역_정리", "정책명_정리"]).sum()),
        int(df.duplicated(subset=["정책명_정리"]).sum()),
    ],
    "처리방침": [
        "발견 시 제거 가능",
        "기본 중복 제거 기준",
        "지역 간 반복 정책일 수 있으므로 제거하지 않음",
        "정책ID가 없을 때만 보조 기준으로 사용",
        "지역 간 반복 정책일 수 있으므로 제거하지 않음",
    ]
})

display(duplicate_report)

,점검기준,중복행수,처리방침
0,행 전체 완전 동일,0,발견 시 제거 가능
1,동일 지역 + 동일 정책ID,0,기본 중복 제거 기준
2,동일 정책ID만 기준,3665,지역 간 반복 정책일 수 있으므로 제거하지 않음
3,동일 지역 + 동일 정책명,221,정책ID가 없을 때만 보조 기준으로 사용
4,동일 정책명만 기준,3836,지역 간 반복 정책일 수 있으므로 제거하지 않음


In [5]:
# 중복 제거: 지역별 정책 공급량 분석을 위해 '지역 + 정책ID' 기준만 제거합니다.
before = len(df)

df["_최종수정일시_dt"] = pd.to_datetime(df["최종수정일시"], errors="coerce")
df = df.sort_values(["지역_정리", "_최종수정일시_dt"], ascending=[True, False])

has_id = df["정책ID_정리"].ne("")
df_with_id = df[has_id].drop_duplicates(subset=["지역_정리", "정책ID_정리"], keep="first")
df_without_id = df[~has_id].drop_duplicates(subset=["지역_정리", "정책명_정리"], keep="first")

df = pd.concat([df_with_id, df_without_id], ignore_index=True)
after = len(df)

print(f"중복 제거 전: {before:,}건")
print(f"중복 제거 후: {after:,}건")
print(f"제거된 중복: {before - after:,}건")

중복 제거 전: 5,470건
중복 제거 후: 5,470건
제거된 중복: 0건


In [6]:
# 정책 분야 재분류
# 기존 대표분류는 참고용으로 보존하고, 공식 대분류/중분류를 우선 반영한 분석용 분류를 새로 만듭니다.
CATEGORY_ORDER = ["일자리", "직무교육", "주거지원", "창업지원", "복지", "참여 프로그램", "기타"]


def normalize_existing_category(row) -> str:
    base = clean_basic_text(row.get("대표분류", "")) or clean_basic_text(row.get("자동분류", ""))
    if "," in base:
        base = base.split(",")[0].strip()

    aliases = {
        "주거": "주거지원",
        "주택": "주거지원",
        "교육": "직무교육",
        "취업교육": "직무교육",
        "창업": "창업지원",
        "참여": "참여 프로그램",
        "프로그램": "참여 프로그램",
        "복지지원": "복지",
        "금융": "복지",
        "문화": "복지",
    }

    if base in CATEGORY_ORDER:
        return base
    for k, v in aliases.items():
        if k in base:
            return v
    return "기타"


def classify_policy_official(row) -> str:
    major = clean_basic_text(row.get("정책대분류", ""))
    mid = clean_basic_text(row.get("정책중분류", ""))
    existing = normalize_existing_category(row)
    official_text = f"{major} {mid}"

    # 중분류가 더 구체적이므로 우선 반영합니다.
    if re.search(r"창업", mid):
        return "창업지원"
    if re.search(r"주거|주택|전월세|임대|거주", official_text):
        return "주거지원"
    if re.search(r"미래역량|교육|훈련|온라인교육|교육비|자격|인재양성|직무", official_text):
        return "직무교육"
    if re.search(r"청년참여|정책인프라|권익보호|국제교류|참여권리|참여[･·]기반|참여", official_text):
        return "참여 프로그램"
    if re.search(r"금융|복지|문화|건강|취약|생활지원|예술인", official_text):
        return "복지"
    if re.search(r"일자리|취업|재직자|고용", official_text):
        return "일자리"

    return existing if existing in CATEGORY_ORDER else "기타"


df["대표분류_기존정리"] = df.apply(normalize_existing_category, axis=1)
df["대표분류_정리"] = df.apply(classify_policy_official, axis=1)
df["분류변경여부"] = (df["대표분류_기존정리"] != df["대표분류_정리"]).astype(int)

print("기존 대표분류 기준")
display(df["대표분류_기존정리"].value_counts().reindex(CATEGORY_ORDER, fill_value=0).rename_axis("분류").reset_index(name="정책수"))

print("개선 대표분류 기준")
display(df["대표분류_정리"].value_counts().reindex(CATEGORY_ORDER, fill_value=0).rename_axis("분류").reset_index(name="정책수"))

print("기존분류 × 개선분류 교차표")
display(pd.crosstab(df["대표분류_기존정리"], df["대표분류_정리"]).reindex(index=CATEGORY_ORDER, columns=CATEGORY_ORDER, fill_value=0))

기존 대표분류 기준


,분류,정책수
0,일자리,3155
1,직무교육,1305
2,주거지원,324
3,창업지원,18
4,복지,585
5,참여 프로그램,83
6,기타,0


개선 대표분류 기준


,분류,정책수
0,일자리,1541
1,직무교육,943
2,주거지원,375
3,창업지원,765
4,복지,1030
5,참여 프로그램,816
6,기타,0


기존분류 × 개선분류 교차표


대표분류_정리,일자리,직무교육,주거지원,창업지원,복지,참여 프로그램,기타
대표분류_기존정리,,,,,,,
일자리,1541,450,45,765,178,176,0
직무교육,0,493,45,0,342,425,0
주거지원,0,0,285,0,27,12,0
창업지원,0,0,0,0,17,1,0
복지,0,0,0,0,466,119,0
참여 프로그램,0,0,0,0,0,83,0
기타,0,0,0,0,0,0,0


In [7]:
# 텍스트마이닝용 텍스트 정리
# 나이조건/소득조건은 지원대상에 이미 포함되는 경우가 많으므로 별도 결합에서 제외합니다.
analysis_text_cols = [
    "정책명", "정책키워드", "정책설명", "정책지원내용",
    "지원대상", "신청방법", "주관기관", "운영기관"
]


def remove_code_noise(text: str) -> str:
    """API 코드성 숫자와 반복적으로 등장하는 형식 문구를 줄입니다. 금액/연도 숫자는 보존합니다."""
    text = clean_basic_text(text)
    # 0043003 0 0, 0011009 등 코드성 토큰 제거
    text = re.sub(r"\b\d{7}\b(?:\s+0){0,3}", " ", text)
    text = re.sub(r"\b0\s+0\b", " ", text)
    text = re.sub(r"연령제한\s*:\s*[YN]", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def join_unique_text(row, cols) -> str:
    """여러 컬럼을 결합하되 완전히 같은 문구는 한 번만 남깁니다."""
    seen = set()
    parts = []
    for col in cols:
        value = remove_code_noise(row.get(col, ""))
        if not value:
            continue
        key = value.lower()
        if key not in seen:
            seen.add(key)
            parts.append(value)
    return clean_basic_text(" ".join(parts))


df["분석텍스트_원본"] = df.apply(lambda row: join_unique_text(row, analysis_text_cols), axis=1)
df["분석텍스트"] = df["분석텍스트_원본"].map(remove_code_noise)
df["분석텍스트길이"] = df["분석텍스트"].str.len()

# 정보 완성도 보조 지표
url_cols = ["신청URL", "참고URL1", "참고URL2"]
df["URL존재여부"] = df[url_cols].apply(lambda row: any(clean_basic_text(x) for x in row), axis=1).astype(int)
df["지원대상존재여부"] = df["지원대상"].str.len().gt(0).astype(int)
df["신청방법존재여부"] = df["신청방법"].str.len().gt(0).astype(int)
df["제출서류존재여부"] = df["제출서류"].str.len().gt(0).astype(int)

print("분석텍스트 길이 요약")
display(df["분석텍스트길이"].describe().to_frame().T)
display(df[["지역", "정책명", "대표분류_정리", "분석텍스트길이", "URL존재여부", "분석텍스트"]].head())

분석텍스트 길이 요약


,count,mean,std,min,25%,50%,75%,max
분석텍스트길이,5470.0,397.182633,304.954502,62.0,192.0,301.0,502.0,2399.0


,지역,정책명,대표분류_정리,분석텍스트길이,URL존재여부,분석텍스트
0,강원,청년미래적금,복지,302,1,"청년미래적금 보조금 청년들의 기초자산 형성을 지원하기 위한 정책형 금융상품으로, 3년 만기 시까지 매월 최대 50만원 한도 내에서 자유롭게 납입 가능(2026년 6월 22일 출시 예정) 은행이자+비과세 혜택+..."
1,강원,스마트 모빌리티 창업캠프사업,창업지원,414,1,"스마트 모빌리티 창업캠프사업 교육지원 미래모빌리티 분야 창업을 희망하는 청년을 대상으로 국내 자동차업계 마이스터들의 강연, 멘토링, 카운슬링을 지원하여 창업아이디어를 구체화하고 창업 준비를 위한 각종 프로그램..."
2,강원,산림산업 창업지원_청년 임팩트 창업 아이디어 챌린지,창업지원,502,1,"산림산업 창업지원_청년 임팩트 창업 아이디어 챌린지 교육지원,맞춤형상담서비스,장기미취업청년 산림자원을 활용 기반의 산림 문제 해결형 사업 공모전 개최 및 실증 과정을 통한 청년 창업 경험 지원 ㅇ 공모전명 :..."
3,강원,(농식품부) 농식품 바우처,복지,731,1,(농식품부) 농식품 바우처 바우처 『농업·농촌 및 식품산업 기본법』 제 23조의 2(취약계층 등에 대한 식품지원)에 근거하여 취약계층의 식품 접근성을 강화하고 균형 있는 식품 섭취를 지원하는 식품지원 제도 -...
4,강원,삼척형 청년인턴 지원사업,일자리,323,1,"삼척형 청년인턴 지원사업 인턴 삼척시 거주 미취업 청년(18~49세) 공공 행정 분야 인턴 연수 기회 제공 상/하반기 행정인턴 모집(주 5일 근무, 1일 7시간) 18세~49세 / □ 선발 우선순위(모집인원 ..."


In [8]:
# 신청기간/사업기간 날짜 파생: 여러 날짜 형식을 지원합니다.
def _parse_date_token(token: str):
    token = clean_basic_text(token)

    # YYYYMMDD
    if re.fullmatch(r"20\d{6}", token):
        return pd.to_datetime(token, format="%Y%m%d", errors="coerce")

    # YYYY-MM-DD, YYYY.MM.DD, YYYY년 M월 D일 등
    m = re.search(r"(20\d{2})\D+(\d{1,2})\D+(\d{1,2})", token)
    if m:
        y, mo, d = map(int, m.groups())
        return pd.to_datetime(f"{y:04d}-{mo:02d}-{d:02d}", errors="coerce")

    return pd.NaT


def extract_period_dates(period_text):
    text = clean_basic_text(period_text)
    if not text:
        return pd.NaT, pd.NaT

    tokens = []
    tokens += re.findall(r"20\d{6}", text)
    tokens += re.findall(r"20\d{2}\D+\d{1,2}\D+\d{1,2}", text)

    dates = []
    for token in tokens:
        dt = _parse_date_token(token)
        if pd.notna(dt):
            dates.append(dt)

    if not dates:
        return pd.NaT, pd.NaT

    return min(dates), max(dates)


def is_always_or_open_period(period_text) -> int:
    text = clean_basic_text(period_text)
    if not text:
        return 0
    return int(bool(re.search(r"상시|연중|계속|수시|예산\s*소진|별도\s*공고", text)))


df[["신청시작일", "신청종료일"]] = df["신청기간"].apply(lambda x: pd.Series(extract_period_dates(x)))
df[["사업시작일", "사업종료일"]] = df["사업기간"].apply(lambda x: pd.Series(extract_period_dates(x)))

today = pd.Timestamp.today().normalize()
df["신청기간_기재여부"] = df["신청기간"].str.len().gt(0).astype(int)
df["신청기간_날짜추출여부"] = df["신청시작일"].notna().astype(int)
df["신청기간_상시성여부"] = df["신청기간"].map(is_always_or_open_period)
df["사업기간_기재여부"] = df["사업기간"].str.len().gt(0).astype(int)
df["사업기간_날짜추출여부"] = df["사업시작일"].notna().astype(int)
df["사업기간_상시성여부"] = df["사업기간"].map(is_always_or_open_period)

df["현재신청가능추정"] = (
    (
        df["신청시작일"].notna()
        & df["신청종료일"].notna()
        & (df["신청시작일"] <= today)
        & (today <= df["신청종료일"])
    )
    | df["신청기간_상시성여부"].eq(1)
).astype(int)

period_quality = pd.DataFrame({
    "항목": ["신청기간", "사업기간"],
    "기재건수": [int(df["신청기간_기재여부"].sum()), int(df["사업기간_기재여부"].sum())],
    "날짜추출건수": [int(df["신청기간_날짜추출여부"].sum()), int(df["사업기간_날짜추출여부"].sum())],
    "상시성표현건수": [int(df["신청기간_상시성여부"].sum()), int(df["사업기간_상시성여부"].sum())],
})

display(period_quality)
display(df[["정책명", "신청기간", "신청시작일", "신청종료일", "현재신청가능추정"]].head())

,항목,기재건수,날짜추출건수,상시성표현건수
0,신청기간,2529,2529,0
1,사업기간,5467,3245,1527


,정책명,신청기간,신청시작일,신청종료일,현재신청가능추정
0,청년미래적금,20260622 ~ 20261231,2026-06-22,2026-12-31,0
1,스마트 모빌리티 창업캠프사업,20260401 ~ 20260528,2026-04-01,2026-05-28,0
2,산림산업 창업지원_청년 임팩트 창업 아이디어 챌린지,20260520 ~ 20260603,2026-05-20,2026-06-03,0
3,(농식품부) 농식품 바우처,20251222 ~ 20261211,2025-12-22,2026-12-11,1
4,삼척형 청년인턴 지원사업,20260202 ~ 20260206,2026-02-02,2026-02-06,0


In [9]:
# 지역별/분류별 요약표
region_summary = (
    df.groupby("지역_정리", as_index=False)
      .agg(
          정책수=("정책명", "count"),
          평균텍스트길이=("분석텍스트길이", "mean"),
          URL존재율=("URL존재여부", "mean"),
          지원대상기재율=("지원대상존재여부", "mean"),
          신청방법기재율=("신청방법존재여부", "mean"),
          신청기간기재율=("신청기간_기재여부", "mean"),
          신청기간날짜추출률=("신청기간_날짜추출여부", "mean"),
          현재신청가능추정비율=("현재신청가능추정", "mean"),
      )
      .rename(columns={"지역_정리": "지역"})
      .sort_values("정책수", ascending=False)
)

category_summary = (
    df.groupby("대표분류_정리", as_index=False)
      .agg(
          정책수=("정책명", "count"),
          평균텍스트길이=("분석텍스트길이", "mean"),
          URL존재율=("URL존재여부", "mean"),
          신청방법기재율=("신청방법존재여부", "mean"),
      )
      .sort_values("정책수", ascending=False)
)

region_category_pivot = (
    pd.pivot_table(
        df,
        index="지역_정리",
        columns="대표분류_정리",
        values="정책명",
        aggfunc="count",
        fill_value=0
    )
    .reindex(columns=CATEGORY_ORDER, fill_value=0)
)
region_category_pivot.index.name = "지역"

preprocessing_quality_summary = pd.DataFrame({
    "항목": [
        "원본 행 수",
        "전처리 후 행 수",
        "제거된 중복 수",
        "분석텍스트 공백 행 수",
        "정책ID 고유 수",
        "동일 정책ID 지역 반복 행 수",
        "분류 변경 행 수",
    ],
    "값": [
        len(df_raw),
        len(df),
        len(df_raw) - len(df),
        int(df["분석텍스트"].str.len().eq(0).sum()),
        int(df["정책ID_정리"].nunique()),
        int(df.duplicated(subset=["정책ID_정리"]).sum()),
        int(df["분류변경여부"].sum()),
    ],
    "해석": [
        "API 수집 원자료의 행 수",
        "지역×정책ID 기준 분석 행 수",
        "동일 지역 내 중복 정책 제거 수",
        "0이면 텍스트마이닝 가능",
        "지역을 제거하고 본 고유 정책 수",
        "전국/복수지역 정책 반복 가능성이 있어 제거하지 않음",
        "공식 대분류/중분류 기준 재분류된 행 수",
    ]
})

display(preprocessing_quality_summary)
display(region_summary)
display(category_summary)
display(region_category_pivot)

,항목,값,해석
0,원본 행 수,5470,API 수집 원자료의 행 수
1,전처리 후 행 수,5470,지역×정책ID 기준 분석 행 수
2,제거된 중복 수,0,동일 지역 내 중복 정책 제거 수
3,분석텍스트 공백 행 수,0,0이면 텍스트마이닝 가능
4,정책ID 고유 수,1805,지역을 제거하고 본 고유 정책 수
5,동일 정책ID 지역 반복 행 수,3665,전국/복수지역 정책 반복 가능성이 있어 제거하지 않음
6,분류 변경 행 수,2602,공식 대분류/중분류 기준 재분류된 행 수


,지역,정책수,평균텍스트길이,URL존재율,지원대상기재율,신청방법기재율,신청기간기재율,신청기간날짜추출률,현재신청가능추정비율
8,충남,707,323.004243,0.816124,1,0.613861,0.437058,0.437058,0.084866
7,제주,608,405.138158,0.800987,1,0.659539,0.495066,0.495066,0.049342
2,경남,584,394.638699,0.833904,1,0.539384,0.414384,0.414384,0.017123
3,경북,539,415.497217,0.788497,1,0.589981,0.504638,0.504638,0.031540
5,전남,535,395.276636,0.753271,1,0.605607,0.454206,0.454206,0.024299
6,전북,534,398.061798,0.720974,1,0.593633,0.464419,0.464419,0.039326
1,경기,515,396.398058,0.811650,1,0.633010,0.473786,0.473786,0.052427
9,충북,498,412.483936,0.751004,1,0.672691,0.437751,0.437751,0.020080
0,강원,489,410.584867,0.754601,1,0.664622,0.488753,0.488753,0.030675
4,서울,461,453.585683,0.750542,1,0.609544,0.462039,0.462039,0.019523


,대표분류_정리,정책수,평균텍스트길이,URL존재율,신청방법기재율
1,일자리,1541,464.159637,0.835821,0.687865
0,복지,1030,396.540777,0.762136,0.580583
3,직무교육,943,392.512195,0.774125,0.670201
4,참여 프로그램,816,336.055147,0.753676,0.588235
5,창업지원,765,310.522876,0.759477,0.541176
2,주거지원,375,445.258667,0.725333,0.512000


대표분류_정리,일자리,직무교육,주거지원,창업지원,복지,참여 프로그램,기타
지역,,,,,,,
강원,146,89,26,72,85,71,0
경기,146,94,32,69,94,80,0
경남,175,94,43,81,102,89,0
경북,158,84,44,86,93,74,0
서울,129,81,29,67,85,70,0
전남,140,95,39,81,100,80,0
전북,152,89,32,76,104,81,0
제주,171,107,45,82,111,92,0
충남,179,118,51,87,169,103,0


In [10]:
# 저장
preprocessed_path = OUTPUT_DIR / "policy_preprocessed.csv"
region_summary_path = OUTPUT_DIR / "policy_region_summary.csv"
category_summary_path = OUTPUT_DIR / "policy_category_summary.csv"
pivot_path = OUTPUT_DIR / "policy_region_category_pivot.csv"
quality_path = OUTPUT_DIR / "policy_preprocessing_quality_summary.csv"
duplicate_report_path = OUTPUT_DIR / "policy_duplicate_diagnostics.csv"

# 보조 정렬용 컬럼은 저장 전에 남겨도 되지만, 보고서용 가독성을 위해 맨 뒤로 정리합니다.
df.to_csv(preprocessed_path, index=False, encoding="utf-8-sig")
region_summary.to_csv(region_summary_path, index=False, encoding="utf-8-sig")
category_summary.to_csv(category_summary_path, index=False, encoding="utf-8-sig")
region_category_pivot.to_csv(pivot_path, encoding="utf-8-sig")
preprocessing_quality_summary.to_csv(quality_path, index=False, encoding="utf-8-sig")
duplicate_report.to_csv(duplicate_report_path, index=False, encoding="utf-8-sig")

print("저장 완료")
print("-", preprocessed_path)
print("-", region_summary_path)
print("-", category_summary_path)
print("-", pivot_path)
print("-", quality_path)
print("-", duplicate_report_path)

저장 완료
- outputs\policy_preprocessed.csv
- outputs\policy_region_summary.csv
- outputs\policy_category_summary.csv
- outputs\policy_region_category_pivot.csv
- outputs\policy_preprocessing_quality_summary.csv
- outputs\policy_duplicate_diagnostics.csv


## 보고서에 쓸 수 있는 전처리 설명 문장

본 연구에서는 온통청년 API로 수집한 청년정책 데이터를 `지역 × 정책ID` 단위로 정리하였다. 동일한 정책ID가 여러 지역에 나타나는 경우는 전국 공통 정책 또는 복수 지역 대상 정책일 수 있으므로 지역별 정책 공급량 분석에서는 제거하지 않았고, 동일 지역 내 동일 정책ID가 반복된 경우만 중복으로 판단하였다. 정책 분야는 기존 자동분류를 그대로 사용하지 않고, 온통청년의 공식 `정책대분류`와 `정책중분류`를 우선 반영하여 일자리, 직무교육, 주거지원, 창업지원, 복지, 참여 프로그램으로 재분류하였다. 또한 텍스트마이닝을 위해 정책명, 키워드, 설명, 지원내용, 지원대상, 신청방법, 기관 정보를 결합하되, 코드성 숫자와 중복 문구를 제거하여 분석 텍스트를 구성하였다.